### RETRIEVAL

In [27]:
from dotenv import load_dotenv
from langchain_openai import ChatOpenAI
from langchain_chroma import Chroma
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_huggingface import HuggingFaceEmbeddings
import gradio as gr
from rich import print
import os

In [28]:
# Loading environment variables from .env file
db_name = "vector_db"

load_dotenv(override=True)

groq_api_key = os.getenv("GROQ_API_KEY")
groq_model = os.getenv("GROQ_MODEL")
groq_base_url = os.getenv("GROQ_BASE_URL")

if groq_api_key:
    print(f"Groq API Key exists: {groq_api_key[:4]}, {groq_model[-8:]}, {groq_base_url[-8:]}")
else:
    print("Groq API Key not set.")

Groq API Key exists: gsk_, -oss-20b, penai/v1

Connect to Chroma, use Hugging Face `all-MiniLM-L6-v2`

In [29]:
embeddings = HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2") # It converts each text chunk into a 384-dimensional coordinate space.
vectorstore = Chroma(persist_directory=db_name, embedding_function=embeddings)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Set up the two key Langchain objects: `retriever` and `llm`

A sidebar on __temprature__:

* Controls how diverse the output is.
* A temprature of 0 means that the output should be predictable.
* Higher temprature for more variety in answers.

Some people describe temprature as being like _creativity_ but that's not quite right.

* It actually controls which tokens get selected during inference.
* __temprature = 0__ means: always select the token with highest probability.
* __temprature = 1__ means: a token with 10% probability should be picked 10% of the time.

__Note:__ If you want creativity, use system prompt.

In [30]:
retriever = vectorstore.as_retriever()
# Pass your groq_model explicitly into the model parameter
llm = ChatOpenAI(
    model=groq_model,
    api_key=groq_api_key, 
    base_url=groq_base_url, 
    temperature=0
)

These LangChain objects implement the method `invoke()`.

In [31]:
print(retriever.invoke("Who is Avery?"))

[
    Document(
        id='049b20a2-6c6b-4194-b415-5dcc9410f6ac',
        metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'},
        page_content="## Other HR Notes\n- **Professional Development**: Avery has actively participated in 
leadership training programs and industry conferences, representing Insurellm and fostering partnerships.  \n- 
**Diversity & Inclusion Initiatives**: Avery has championed a commitment to diversity in hiring practices, seeing 
visible improvements in team representation since 2021.  \n- **Work-Life Balance**: Feedback revealed concerns 
regarding work-life balance, which Avery has approached by implementing flexible working conditions and ensuring 
regular check-ins with the team.\n- **Community Engagement**: Avery led community outreach efforts, focusing on 
financial literacy programs, particularly aimed at underserved populations, improving Insurellm's corporate social 
responsibility image.  \n\nAvery Lancaster has demonstrated resilience and adaptability throughout her career at 
Insurellm, positioning the company as a key player in the insurance technology landscape."
    ),
    Document(
        id='0ae7a12b-35fb-4a39-8490-d2b766ca285b',
        metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'},
        page_content='- **2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing 
employee concerns, leading to overall improvement despite a saturated market.  \n\n- **2023**: **Exceeds 
Expectations**  \n  Market leadership was regained with innovative approaches to personalized insurance solutions. 
Avery is now recognized in industry publications as a leading voice in Insurance Tech innovation.'
    ),
    Document(
        id='e5e88f58-3a68-4a96-a226-72cc5c42a165',
        metadata={'source': 'knowledge-base\\employees\\Avery Lancaster.md', 'doc_type': 'employees'},
        page_content="- **2018**: **Exceeds Expectations**  \n  Under Avery’s pivoted vision, Insurellm launched 
two new successful products that significantly increased market share.  \n\n- **2019**: **Meets Expectations**  \n 
Steady growth, however, some team tensions led to a minor drop in employee morale. Avery recognized the need to 
enhance company culture.  \n\n- **2020**: **Below Expectations**  \n  The COVID-19 pandemic posed unforeseen 
operational difficulties. Avery faced criticism for delayed strategy shifts, although efforts were eventually made 
to stabilize the company.  \n\n- **2021**: **Exceptional**  \n  Avery's decisive transition to remote work and 
rapid adoption of digital tools led to record-high customer satisfaction levels and increased sales.  \n\n- 
**2022**: **Satisfactory**  \n  Avery focused on rebuilding team dynamics and addressing employee concerns, leading
to overall improvement despite a saturated market."
    ),
    Document(
        id='b3316e2f-1f75-4d99-8f75-fb72e859ca29',
        metadata={'doc_type': 'employees', 'source': 'knowledge-base\\employees\\Avery Lancaster.md'},
        page_content='# Avery Lancaster\n\n## Summary\n- **Date of Birth**: March 15, 1985\n- **Job Title**: 
Co-Founder & Chief Executive Officer (CEO)\n- **Location**: San Francisco, California\n- **Current Salary**: 
$225,000  \n\n## Insurellm Career Progression\n- **2015 - Present**: Co-Founder & CEO  \n  Avery Lancaster 
co-founded Insurellm in 2015 and has since guided the company to its current position as a leading Insurance Tech 
provider. Avery is known for her innovative leadership strategies and risk management expertise that have 
catapulted the company into the mainstream insurance market.  \n\n- **2013 - 2015**: Senior Product Manager at 
Innovate Insurance Solutions  \n  Before launching Insurellm, Avery was a leading Senior Product Manager at 
Innovate Insurance Solutions, where she developed groundbreaking insurance products aimed at the tech sector.'
    )
]

In [32]:
print(llm.invoke("Who is Avery?"))

AIMessage(
    content='I’m not sure which Avery you’re referring to—there are many people, characters, and even places named 
Avery. Could you let me know a bit more about the context? For example, are you asking about a public figure, a 
fictional character, a historical person, or something else entirely? That’ll help me give you the most accurate 
answer.',
    additional_kwargs={'refusal': None},
    response_metadata={
        'token_usage': {
            'completion_tokens': 133,
            'prompt_tokens': 75,
            'total_tokens': 208,
            'completion_tokens_details': {
                'accepted_prediction_tokens': None,
                'audio_tokens': None,
                'reasoning_tokens': 53,
                'rejected_prediction_tokens': None
            },
            'prompt_tokens_details': None,
            'queue_time': 0.0467133,
            'prompt_time': 0.00424632,
            'completion_time': 0.137331484,
            'total_time': 0.141577804
        },
        'model_provider': 'openai',
        'model_name': 'openai/gpt-oss-20b',
        'system_fingerprint': 'fp_80501ff3a1',
        'id': 'chatcmpl-f67ff644-be30-4607-821d-ed2786091e73',
        'service_tier': 'on_demand',
        'finish_reason': 'stop',
        'logprobs': None
    },
    id='lc_run--019eef09-569a-7d02-bfe5-a1f5d3679958-0',
    tool_calls=[],
    invalid_tool_calls=[],
    usage_metadata={
        'input_tokens': 75,
        'output_tokens': 133,
        'total_tokens': 208,
        'input_token_details': {},
        'output_token_details': {'reasoning': 53}
    }
)

In [33]:
SYSTEM_PROMPT = """
You are a knowledgeable, friendly assistant representing the company Insurellm.
You are chatting with a user about Insurellm.
If relevant, use the given context to answer any question.
If you don't know the answer, say so.
Context:
{context}
"""

In [34]:
def answer_question(question: str, history):
    docs = retriever.invoke(question)
    context = "\n\n".join(doc.page_content for doc in docs)
    system_prompt = SYSTEM_PROMPT.format(context=context)
    response = llm.invoke([SystemMessage(content=system_prompt), HumanMessage(content=question)])
    return response.content

In [37]:
print(answer_question("Who is Averi Lancaster?", []))

It looks like you’re asking about **Avery Lancaster** (sometimes misspelled as “Averi”). Here’s a quick snapshot of
who she is:

| Detail | Information |
|--------|-------------|
| **Full Name** | Avery Lancaster |
| **Date of Birth** | March 15, 1985 |
| **Current Role** | Co‑Founder & Chief Executive Officer (CEO) of Insurellm |
| **Location** | San Francisco, California |
| **Current Salary** | $225,000 |
| **Career Highlights** | • **2015‑Present** – Co‑Founder & CEO: Built Insurellm into a leading InsurTech provider,
driving innovation and market expansion.<br>• **2013‑2015** – Senior Product Manager at Innovate Insurance 
Solutions: Developed cutting‑edge insurance products for the tech sector.<br>• **Jan 2021‑Present** – Senior Data 
Engineer (note: this appears to be a separate role mentioned in the context; it may be a typo or a dual role). |
| **Key Achievements** | • Recognized as Insurellm Innovator of the Year in 2023 (IIOTY 2023 award).<br>• Known for
innovative leadership, risk‑management expertise, and steering the company into mainstream insurance markets. |

If you need more details—such as her leadership style, strategic initiatives, or how she’s shaped Insurellm’s 
product roadmap—just let me know!

In [ ]:
gr.ChatInterface(answer_question, title="Insurellm Chatbot", description="Ask me anything about Insurellm!").launch()

* Running on local URL:  http://127.0.0.1:7867
* To create a public link, set `share=True` in `launch()`.


d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await queue_join_helper(body, request, username)
d:\ujjwal\expert_knowledge_worker\.venv\Lib\site-packages\gradio\routes.py:1379: StarletteDeprecationWarning: 'HTTP_422_UNPROCESSABLE_ENTITY' is deprecated. Use 'HTTP_422_UNPROCESSABLE_CONTENT' instead.
  return await